# Iterative Methods

A discretised differential equation gives a linear system with many unknowns and few nonzeros in each row. A direct solver factorises the matrix; an iterative method only multiplies it by vectors. This notebook solves such systems with the Krylov methods of Krylov.jl: CG, MINRES, GMRES and LSQR, the residual at every iteration, a preconditioner, and how the iterations grow as the grid is refined.

In [ ]:
Pollis.packages("Plots", "Krylov", "AlgebraicMultigrid")

In [ ]:
using LinearAlgebra, SparseArrays, Random, Plots, Krylov

## A large sparse matrix

The five-point Laplacian on an $n \times n$ grid of the unit square, with $u = 0$ on the boundary, has $n^2$ unknowns and at most five nonzeros in each row. It is symmetric positive definite.

In [ ]:
function laplacian(n)
    h = 1 / (n + 1)
    T = spdiagm(-1 => fill(-1.0, n - 1), 0 => fill(2.0, n), 1 => fill(-1.0, n - 1)) / h^2
    Id = sparse(1.0I, n, n)
    return kron(Id, T) + kron(T, Id)
end
L = laplacian(50)
(size = size(L), nonzeros = nnz(L), share_of_entries = nnz(L) / length(L))

In [ ]:
spy(L, legend=false, title="Nonzeros of the Laplacian, 50 x 50 grid")

## Conjugate gradients

CG needs only products `L * v`. With `history = true` it records the residual norm at every iteration. The tolerance is relative: stop when the residual is $10^{-8}$ of the norm of $b$.

In [ ]:
b = ones(size(L, 1))
x_cg, stats = cg(L, b; atol = 0.0, rtol = 1e-8, history = true)
x_dir = L \ b
(iterations = stats.niter, solved = stats.solved, relative_residual = norm(b - L * x_cg) / norm(b), difference_from_direct = norm(x_cg - x_dir) / norm(x_dir))

In [ ]:
plot(0:stats.niter, stats.residuals ./ norm(b), yscale=:log10, lw=2, label="CG", xlabel="iteration", ylabel="residual / norm(b)")

In [ ]:
heatmap(reshape(x_cg, 50, 50), aspect_ratio=:equal, c=:viridis, title="Solution on the grid")

## Iterations and the grid

The bound on the CG error depends on $\sqrt{\kappa}$, and the condition number of the Laplacian grows as the grid is refined. Watch the iteration count as the number of points per side doubles, beside the time of CG and of the sparse direct solve.

In [ ]:
map((25, 50, 100, 200)) do n
    Ln = laplacian(n); bn = ones(n^2)
    _, s = cg(Ln, bn; atol = 0.0, rtol = 1e-8)
    t_dir = @elapsed Ln \ bn
    (points_per_side = n, unknowns = n^2, cg_iterations = s.niter, cg_seconds = s.timer, direct_seconds = t_dir)
end

## A preconditioner

Algebraic multigrid builds a hierarchy of coarser problems from the entries of the matrix. Used as a preconditioner (`M`, applied by solving, so `ldiv = true`), it changes how the iteration count grows with the grid. Compare with the table above.

In [ ]:
using AlgebraicMultigrid
map((25, 50, 100, 200)) do n
    Ln = laplacian(n); bn = ones(n^2)
    P = aspreconditioner(ruge_stuben(Ln))
    _, s = cg(Ln, bn; M = P, ldiv = true, atol = 0.0, rtol = 1e-8)
    (points_per_side = n, iterations_with_multigrid = s.niter, seconds = s.timer)
end

## Matrices that are not symmetric positive definite

Shift the Laplacian past its smallest eigenvalues and it becomes indefinite; add convection and it becomes nonsymmetric. Each structure has its method: MINRES for symmetric indefinite matrices, GMRES for nonsymmetric ones. The table also runs CG on all three, to see what happens when its assumptions fail.

In [ ]:
n = 50; h = 1 / (n + 1)
D = spdiagm(-1 => fill(-1.0, n - 1), 1 => fill(1.0, n - 1)) / (2h)
matrices = ["symmetric positive definite" => L, "symmetric indefinite" => L - 110.0I, "nonsymmetric" => L + 50.0 * kron(sparse(1.0I, n, n), D)]
rows = []
for (kind, A) in matrices, (name, solver) in ("CG" => cg, "MINRES" => minres, "GMRES" => gmres)
    xs, s = solver(A, b; atol = 0.0, rtol = 1e-8)
    push!(rows, (matrix = kind, method = name, iterations = s.niter, solved = s.solved, relative_residual = norm(b - A * xs) / norm(b)))
end
rows

## Least squares

Stack the Laplacian on the first differences of the grid along $x$ and $y$: more equations than unknowns. LSQR and LSMR minimise $\lVert b - Ax \rVert$. The residual itself does not go to zero; what goes to zero is $\lVert A^\top r \rVert$, the residual of the normal equations. LSMR makes that quantity decrease monotonically.

In [ ]:
G = spdiagm(51, 50, 0 => fill(1.0, 50), -1 => fill(-1.0, 50)) * 51   # first differences between neighbours, 51 x 50
Id = sparse(1.0I, 50, 50)
A = [L; kron(Id, G); kron(G, Id)]   # the Laplacian and the differences along x and y: 7600 equations, 2500 unknowns
bt = ones(size(A, 1))
x1, s1 = lsqr(A, bt; axtol = 1e-8, btol = 1e-8, history = true)
x2, s2 = lsmr(A, bt; axtol = 1e-8, btol = 1e-8, history = true)
x_qr = A \ bt   # sparse QR
(lsqr_iterations = s1.niter, lsmr_iterations = s2.niter, lsqr_difference = norm(x1 - x_qr) / norm(x_qr), lsmr_difference = norm(x2 - x_qr) / norm(x_qr))

In [ ]:
plot(s1.Aresiduals ./ s1.Aresiduals[1], yscale=:log10, lw=2, label="LSQR", xlabel="iteration", ylabel="normal-equations residual, relative")
plot!(s2.Aresiduals ./ s2.Aresiduals[1], lw=2, label="LSMR")

## References

[Hestenes and Stiefel (1952)](https://doi.org/10.6028/jres.049.044), conjugate gradients; [Paige and Saunders (1975)](https://doi.org/10.1137/0712047), MINRES; [Saad and Schultz (1986)](https://doi.org/10.1137/0907058), GMRES; [Paige and Saunders (1982)](https://doi.org/10.1145/355984.355989), LSQR; [Fong and Saunders (2011)](https://doi.org/10.1137/10079687X), LSMR; [Ruge and Stüben (1987)](https://doi.org/10.1137/1.9781611971057.ch4), algebraic multigrid; [Saad (2003)](https://doi.org/10.1137/1.9780898718003), *Iterative Methods for Sparse Linear Systems*, SIAM; [Montoison and Orban (2023)](https://doi.org/10.21105/joss.05187), Krylov.jl.